# M2: NGCF의 CLV 수준·구성·가격 좌표 임베딩 (Dunnhumby, seed 42)

LightGCN에서 상대적으로 가장 유망했던 동일한 historical-CLV layer-0 입력을 NGCF에 이식합니다. `NGCF@64`, 동일 총차원의 `NGCF@67`, 실제 CLV, degree-matched CLV 순열을 고정 100 epoch로 비교합니다. 최종 test와 holdout은 구성하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
SOURCE_COMMIT = '__SOURCE_COMMIT__'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
if len(SOURCE_COMMIT) != 40:
    raise RuntimeError('검토된 소스 커밋을 SOURCE_COMMIT에 고정해야 합니다')
!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch origin; else git clone {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} checkout {SOURCE_COMMIT}
%cd {REPO_DIR}
!git rev-parse HEAD

In [ ]:
import json
import torch
from ngcf_clv_level_composition_price_screen import (
    configure_ngcf_clv_screen,
    preflight_summary,
    run_ngcf_clv_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_ngcf_clv_screen()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_ngcf_clv_screen(cfg)

In [ ]:
import pandas as pd
from IPython.display import display

print('1) 절대지표: LightGCN 참고, NGCF@64, NGCF@67, 실제 CLV, degree-matched shuffle')
display(result_df)
print('2) NGCF 내부 대조군별 성과 비교')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) NGCF@67 및 shuffle 대비 실제 CLV Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))